In [20]:
import subprocess
subprocess.run(['pip', 'install', 'pymupdf', 'groq', '--quiet'])
print('Done')

Done


In [ ]:
import os
import re
import json
import time
import fitz
import pandas as pd
from pathlib import Path
from groq import Groq

PDF_FOLDER   = Path('../data/msrit_pdfs')
SYLLABUS_CSV = Path('../data/syllabus_dataset.csv')
GROQ_MODEL   = 'llama-3.1-8b-instant'
GROQ_API_KEY = os.environ.get('GROQ_API_KEY')
INSTITUTE    = 'MSRIT'

groq_client = Groq(api_key=GROQ_API_KEY)

industry = pd.read_csv('../data/industry_dataset.csv')
INDUSTRY_SKILLS = sorted(industry['Skill'].str.lower().str.strip().unique().tolist())

existing = pd.read_csv(SYLLABUS_CSV)
other_institutes = existing[existing['Institute'].str.upper() != 'MSRIT'].copy()

print(f'Industry skills: {len(INDUSTRY_SKILLS)}')
print(f'Existing syllabus rows (other institutes): {len(other_institutes)}')
print(f'Institutes kept: {other_institutes["Institute"].unique()}')

pdf_files = sorted(PDF_FOLDER.glob('*.pdf')) if PDF_FOLDER.exists() else []
print(f'\nPDFs found: {len(pdf_files)}')
for p in pdf_files:
    print(f'  {p.name}')

Industry skills: 95
Existing syllabus rows (other institutes): 260
Institutes kept: ['RVCE' 'BIT' 'PESIT' 'DSCE']

PDFs found: 3
  2nd yr.pdf
  3rd yr.pdf
  4th yr.pdf


In [ ]:
def extract_pages(pdf_path: Path) -> list[str]:
    """Extract text from each page of a PDF separately."""
    doc   = fitz.open(str(pdf_path))
    pages = [page.get_text().strip() for page in doc]
    doc.close()

    pages = [p for p in pages if len(p) > 100]
    return pages


def chunk_pages(pages: list[str], chunk_size: int = 3) -> list[str]:
    """
    Group pages into chunks to send to LLM.
    Groq has token limits so we send a few pages at a time.
    chunk_size=3 means 3 pages per LLM call.
    """
    chunks = []
    for i in range(0, len(pages), chunk_size):
        chunk = '\n\n--- PAGE BREAK ---\n\n'.join(pages[i:i+chunk_size])
        chunks.append(chunk)
    return chunks

all_pages = []
for pdf_path in pdf_files:
    pages = extract_pages(pdf_path)
    print(f'{pdf_path.name}: {len(pages)} pages extracted')
    all_pages.extend(pages)

all_chunks = chunk_pages(all_pages, chunk_size=3)
print(f'\nTotal pages: {len(all_pages)}')
print(f'Total LLM calls needed: {len(all_chunks)}')
print()
print('Sample page text (first 500 chars):')
print(all_pages[0][:500] if all_pages else 'No pages found')

2nd yr.pdf: 38 pages extracted
3rd yr.pdf: 83 pages extracted
4th yr.pdf: 38 pages extracted

Total pages: 159
Total LLM calls needed: 53

Sample page text (first 500 chars):
16 
 
Course Content 
Unit I 
Introduction to Digital Design: Introduction to binary logic, basic theorems, Boolean functions and 
simplification. The Map Method for simplifying Boolean expressions, Two, Three and Four-Variable Map, 
Don’t-Care Conditions, NAND and NOR Implementation, Exclusive-OR Function, Hardware Description 
Language – Verilog code for simple circuits. 
 
Pedagogy/Course Delivery tools: Chalk and talk, Power Point Presentation 
 
Link: https://onlinecourses.nptel.ac.in/noc


In [ ]:
SYSTEM_PROMPT = """You are an expert at reading university syllabus documents and extracting structured information.
Your job is to extract course/subject information from syllabus text and identify which technical skills are taught.

You must output ONLY a valid JSON array. No explanation, no markdown, no extra text.
Each item in the array must have exactly these fields:
  institute: always "MSRIT"
  semester: integer (3 to 8), or 0 if not mentioned
  course: the subject/course name as written in the syllabus
  skill: ONE specific technical skill taught in that course

Rules:
- Extract multiple rows per course (one row per skill)
- Skills must be specific technical terms, not vague phrases
- If semester is not clearly mentioned, use 0
- Output ONLY the JSON array, nothing else"""


def build_user_prompt(chunk_text: str, industry_skills: list) -> str:
    skills_ref = ', '.join(industry_skills[:50])
    return (
        f"Extract all course-skill pairs from this syllabus text.\n\n"
        f"For reference, examples of relevant technical skills include:\n"
        f"{skills_ref}\n\n"
        f"SYLLABUS TEXT:\n"
        f"{chunk_text[:4000]}\n\n"
        f"Output ONLY a JSON array like:\n"
        f'[{{"institute":"MSRIT","semester":5,"course":"Computer Networks","skill":"networking"}}]'
    )


def extract_skills_with_llm(chunk_text: str) -> list[dict]:
    """Send one chunk to Groq LLM and parse the JSON response."""
    try:
        response = groq_client.chat.completions.create(
            model=GROQ_MODEL,
            messages=[
                {'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user',   'content': build_user_prompt(chunk_text, INDUSTRY_SKILLS)}
            ],
            temperature=0.0,
            max_tokens=2048
        )

        raw = response.choices[0].message.content.strip()

        raw = re.sub(r'^```[a-z]*\n?', '', raw)
        raw = re.sub(r'```$', '', raw).strip()

        parsed = json.loads(raw)

        valid = []
        for item in parsed:
            if all(k in item for k in ['institute', 'semester', 'course', 'skill']):
                item['institute'] = 'MSRIT'
                item['skill']     = str(item['skill']).lower().strip()
                item['course']    = str(item['course']).strip()
                item['semester']  = int(item['semester']) if str(item['semester']).isdigit() else 0
                valid.append(item)

        return valid

    except json.JSONDecodeError as e:
        print(f'  JSON parse error: {e}')
        print(f'  Raw response: {raw[:300]}')
        return []
    except Exception as e:
        print(f'  LLM call failed: {e}')
        return []


print('LLM extraction function ready')
print(f'Model: {GROQ_MODEL}')

LLM extraction function ready
Model: llama-3.1-8b-instant


In [ ]:
all_rows = []
failed   = []

print(f'Processing {len(all_chunks)} chunks...\n')

for i, chunk in enumerate(all_chunks):
    print(f'Chunk {i+1}/{len(all_chunks)}...', end=' ')
    rows = extract_skills_with_llm(chunk)
    print(f'{len(rows)} rows extracted')
    all_rows.extend(rows)

    if not rows:
        failed.append(i)

    if i < len(all_chunks) - 1:
        time.sleep(6)

print(f'\nDone.')
print(f'Total rows extracted: {len(all_rows)}')
print(f'Failed chunks: {len(failed)} {failed if failed else ""}')

Processing 53 chunks...

Chunk 1/53... 22 rows extracted
Chunk 2/53... 15 rows extracted
Chunk 3/53... 19 rows extracted
Chunk 4/53... 9 rows extracted
Chunk 5/53... 11 rows extracted
Chunk 6/53... 7 rows extracted
Chunk 7/53... 46 rows extracted
Chunk 8/53... 28 rows extracted
Chunk 9/53... 12 rows extracted
Chunk 10/53... 10 rows extracted
Chunk 11/53... 3 rows extracted
Chunk 12/53... 16 rows extracted
Chunk 13/53... 18 rows extracted
Chunk 14/53... 52 rows extracted
Chunk 15/53... 40 rows extracted
Chunk 16/53...   JSON parse error: Unterminated string starting at: line 1 column 8951 (char 8950)
  Raw response: [{"institute":"MSRIT","semester":5,"course":"Computer Networks","skill":"networking"},{"institute":"MSRIT","semester":5,"course":"Computer Networks","skill":"routing"},{"institute":"MSRIT","semester":5,"course":"Computer Networks","skill":"switching"},{"institute":"MSRIT","semester":5,"course":"Comp
0 rows extracted
Chunk 17/53... 9 rows extracted
Chunk 18/53... 42 rows extr

In [ ]:
recovered_rows = []

for idx in failed:
    print(f"\nRecovering chunk {idx}...")

    chunk = all_chunks[idx]

    rows = extract_skills_with_llm(chunk)

    if rows:
        print(f"Recovered from retry: {len(rows)} rows")
        recovered_rows.extend(rows)
        continue

    print("Retry failed. Splitting chunk...")

    sub_chunks = chunk.split('--- PAGE BREAK ---\n\n')

    chunk_recovered = False
    total_from_chunk = 0

    for i, sub in enumerate(sub_chunks):
        print(f"   Sub-chunk {i+1}/{len(sub_chunks)}...", end=" ")

        sub_rows = extract_skills_with_llm(sub)

        print(f"{len(sub_rows)} rows")

        if sub_rows:
            recovered_rows.extend(sub_rows)
            total_from_chunk += len(sub_rows)
            chunk_recovered = True

    if chunk_recovered:
        print(f"✅ Recovered {total_from_chunk} rows from chunk {idx}")
    else:
        print(f"❌ Completely failed: chunk {idx}")


Recovering chunk 15...
  JSON parse error: Unterminated string starting at: line 1 column 8951 (char 8950)
  Raw response: [{"institute":"MSRIT","semester":5,"course":"Computer Networks","skill":"networking"},{"institute":"MSRIT","semester":5,"course":"Computer Networks","skill":"routing"},{"institute":"MSRIT","semester":5,"course":"Computer Networks","skill":"switching"},{"institute":"MSRIT","semester":5,"course":"Comp
Retry failed. Splitting chunk...
   Sub-chunk 1/3... 17 rows
   Sub-chunk 2/3... 47 rows
   Sub-chunk 3/3... 47 rows
✅ Recovered 111 rows from chunk 15

Recovering chunk 20...
  JSON parse error: Unterminated string starting at: line 1 column 8783 (char 8782)
  Raw response: [{"institute":"MSRIT","semester":0,"course":"Data Analysis using PySpark","skill":"data structures"},{"institute":"MSRIT","semester":0,"course":"Data Analysis using PySpark","skill":"data visualization"},{"institute":"MSRIT","semester":0,"course":"Data Analysis using PySpark","skill":"machine learn

In [35]:
all_rows.extend(recovered_rows)

print("\nFinal total rows:", len(all_rows))


Final total rows: 1265


In [ ]:
msrit_df = pd.DataFrame(all_rows, columns=['institute','semester','course','skill'])

msrit_df.columns = ['Institute','Semester','Course','Skill']

msrit_df = msrit_df.drop_duplicates(subset=['Course','Skill'])
msrit_df = msrit_df[msrit_df['Skill'].str.len() > 1]

print(f'MSRIT rows after dedup: {len(msrit_df)}')
print(f'Unique courses: {msrit_df["Course"].nunique()}')
print(f'Unique skills: {msrit_df["Skill"].nunique()}')
print()
print('Skills extracted:')
print(sorted(msrit_df['Skill'].unique()))
print()
print('Per course breakdown:')
print(msrit_df.groupby('Course')['Skill'].apply(list).to_string())

MSRIT rows after dedup: 1095
Unique courses: 214
Unique skills: 896

Skills extracted:
['a* algorithm', 'a* search', 'abstract class vs interface', 'abstract factory pattern', 'abstraction', 'acceptance testing', 'access control', 'access control in linux', 'accessing memory', 'accountability', 'accuracy', 'acid and base', 'activation functions', 'adaptation from development to production', 'adapter design pattern', 'adapter pattern', 'adders-subtractor', 'advanced data structures', 'advanced features', 'advanced sql', 'adversarial search', 'agents and environments', 'agile planning', 'agile testing principles', 'alexnet', 'algorithm design', 'alu-design', 'alu-operations', 'amortized analysis', 'anomaly-detection', 'ansible configuration', 'apache spark', 'api gateway', 'api rate limiting', 'api testing', 'applications', 'approximation algorithms', 'architecture of arm cortex-m', 'arm 7 architecture', 'arm design philosophy', 'arm embedded systems', 'arm7 instruction set', 'arm7 pipel

In [ ]:
msrit_skill_set = set(msrit_df['Skill'].str.lower().unique())
industry_set    = set(INDUSTRY_SKILLS)

covered  = msrit_skill_set & industry_set
gaps     = industry_set - msrit_skill_set
extra    = msrit_skill_set - industry_set

print(f'Industry skills total        : {len(industry_set)}')
print(f'Covered by MSRIT (from PDF)  : {len(covered)}')
print(f'Gaps (not in MSRIT)          : {len(gaps)}')
print(f'Extra (not in industry list) : {len(extra)}')
print()
print('Covered:')
print(sorted(covered))
print()
print('Gaps (these will appear as curriculum gaps in analysis):')
print(sorted(gaps))

Industry skills total        : 95
Covered by MSRIT (from PDF)  : 58
Gaps (not in MSRIT)          : 37
Extra (not in industry list) : 838

Covered:
['advanced sql', 'api gateway', 'api rate limiting', 'auto scaling', 'aws', 'aws ec2', 'azure', 'background job processing', 'blue-green deployment', 'canary deployment', 'ci/cd', 'ci/cd pipelines', 'circuit breaker pattern', 'cloud architecture', 'cloud cost optimization', 'cloud security', 'css', 'data structures', 'data visualization', 'deep learning', 'disaster recovery planning', 'distributed systems', 'docker', 'event-driven architecture', 'excel', 'feature engineering', 'feature stores', 'generative ai', 'github actions', 'gitops', 'golang', 'grpc', 'helm charts', 'html', 'hyperparameter tuning', 'infrastructure as code', 'java', 'javascript', 'jwt authentication', 'kpi tracking', 'kubernetes', 'lighthouse audits', 'linux', 'llm', 'llm apis', 'llm fine-tuning', 'load balancing', 'machine learning', 'message queues', 'microservices', '

In [ ]:
manual_rows = []

if manual_rows:
    manual_df = pd.DataFrame(manual_rows)
    msrit_df  = pd.concat([msrit_df, manual_df], ignore_index=True)
    msrit_df  = msrit_df.drop_duplicates(subset=['Course','Skill'])
    print(f'Added {len(manual_rows)} manual rows')
else:
    print('No manual additions')

No manual additions


In [ ]:
final_df = pd.concat([other_institutes, msrit_df], ignore_index=True)
final_df = final_df.drop_duplicates()
final_df['Skill']     = final_df['Skill'].str.lower().str.strip()
final_df['Institute'] = final_df['Institute'].str.upper().str.strip()

print('=== FINAL SYLLABUS DATASET ===')
print('Shape:', final_df.shape)
print()
print('Rows per institute:')
print(final_df['Institute'].value_counts())
print()
print('Unique skills per institute:')
print(final_df.groupby('Institute')['Skill'].nunique().sort_values(ascending=False))

final_df.to_csv(SYLLABUS_CSV, index=False)
print(f'\nSaved to {SYLLABUS_CSV}')
print('Run notebook 01 onwards now.')

=== FINAL SYLLABUS DATASET ===
Shape: (1355, 4)

Rows per institute:
Institute
MSRIT    1095
PESIT      74
DSCE       67
BIT        64
RVCE       55
Name: count, dtype: int64

Unique skills per institute:
Institute
MSRIT    896
PESIT     74
DSCE      66
BIT       64
RVCE      54
Name: Skill, dtype: int64

Saved to ..\data\syllabus_dataset.csv
Run notebook 01 onwards now.
